# State Management with StateMachine
This notebook demonstrates how to work with state management using a StateMachine implementation. We'll explore how to create, manage, and control workflow states in a structured way.

## What we'll learn:
- Basic state machine concepts and implementation
- Creating and connecting workflow steps
- Managing state transitions and data flow
- Working with routing and loops in state machines
- Understanding state snapshots and execution flow

### Setup

In [37]:
from typing import TypedDict
from lib.state_machine import (
    StateMachine,
    Step,
    EntryPoint,
    Termination,
)

## Basic State Machine Concepts
Let's start with a simple example that demonstrates the core concepts of our state machine:
1. Defining state schema
2. Creating steps
3. Connecting steps
4. Running the workflow

**Creating the Schema and the State Machine**

In [38]:
class Schema(TypedDict):
    """Schema defining the structure of our state.
    
    Attributes:
        input: The input value to process
        output: The processed output value
    """
    input: int
    output: int

In [39]:
# Create our state machine instance
workflow = StateMachine(Schema)

**Defining the logic for Steps**

In [40]:
def step_input(state: Schema) -> Schema:
    """First step: Increment the input value.
    
    Args:
        state: Current state containing input value
        
    Returns:
        Updated state with incremented value in output
    """
    return {"output": state["input"] + 1, "random": 10}



In [41]:
def step_double(state: Schema) -> Schema:
    """Second step: Double the previous output.
    
    Args:
        state: Current state containing output from previous step
        
    Returns:
        Updated state with doubled output value
    """
    return {"output": state["output"] * 2}


**Creating and Connecting Steps**

In [42]:
entry = EntryPoint()
s1 = Step("input", step_input)
s2 = Step("double", step_double)
termination = Termination()

In [43]:
workflow.add_steps([entry, s1, s2, termination])

In [44]:
workflow.connect(entry, s1)
workflow.connect(s1, s2)
workflow.connect(s2, termination)

In [45]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['input'])],
 'input': [Transition('input' -> ['double'])],
 'double': [Transition('double' -> ['__termination__'])]}

**Running the Workflow**

In [46]:
initial_state = {"input": 4}
workflow = StateMachine(Schema)

workflow.add_steps([entry, s1, s2, termination])
workflow.connect(entry, s1)
workflow.connect(s1, s2)
workflow.connect(s2, termination)

run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: input
[StateMachine] Executing step: double
[StateMachine] Terminating: __termination__


Run('d54a5dcc-cfb3-4c2c-971a-0124d7d5a7f0')

In [47]:
run_object.snapshots

[Snapshot('804e184a-a0a4-42cd-a3e8-1b04a22a5378') @ [2026-10-03 21:58:11.851670]: __entry__.State({'input': 4}),
 Snapshot('93cc5e57-a5a7-4b61-b434-620732547ae5') @ [2026-10-03 21:58:11.851773]: input.State({'input': 4, 'output': 5}),
 Snapshot('74244806-bdcf-470f-9187-6803f13e8ebe') @ [2026-10-03 21:58:11.851841]: double.State({'input': 4, 'output': 10})]

## Advanced State Management: Routing and Loops
Now we'll explore more complex state management patterns including:
- Conditional routing between steps
- Creating loops in the workflow
- Managing state through multiple iterations

In [48]:
class CounterSchema(TypedDict):
    """Schema for a counter-based workflow.
    
    Attributes:
        count: Current counter value
        max_value: Maximum value before termination
    """
    count: int
    max_value: int

In [49]:
workflow = StateMachine(CounterSchema)

In [50]:
def increment_counter(state: CounterSchema) -> CounterSchema:
    """Increment the counter value.
    
    Args:
        state: Current state with counter value
        
    Returns:
        Updated state with incremented counter
    """
    return {"count": state["count"] + 1}

In [51]:
# Create steps
entry = EntryPoint()
increment = Step("increment", increment_counter)
termination = Termination()

In [52]:
workflow.add_steps([entry, increment, termination])

In [53]:
# Router logic
def check_counter(state: CounterSchema) -> Step:
    """Determine next step based on counter value.
    
    Args:
        state: Current state with counter and max value
        
    Returns:
        Next step to execute (increment or terminate)
    """
    if state["count"] >= state["max_value"]:
        return termination
    return increment

In [54]:
# Connect steps with a loop in increment
workflow.connect(entry, increment)
workflow.connect(increment, [increment, termination], check_counter)

In [55]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['increment'])],
 'increment': [Transition('increment' -> ['increment', '__termination__'])]}

In [56]:
initial_state = {"count": 0, "max_value": 3}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Terminating: __termination__


Run('e5902736-54b4-4f33-adb2-cef5c1aac637')

In [57]:
run_object.snapshots

[Snapshot('834647ea-14b7-4782-8c18-618381a20bab') @ [2026-10-03 21:58:12.036071]: __entry__.State({'count': 0, 'max_value': 3}),
 Snapshot('eba682c1-28c8-482c-8027-c545fbc02282') @ [2026-10-03 21:58:12.036203]: increment.State({'count': 1, 'max_value': 3}),
 Snapshot('5ab8a32f-f04b-4fed-893e-d6f799d4a5eb') @ [2026-10-03 21:58:12.036302]: increment.State({'count': 2, 'max_value': 3}),
 Snapshot('052ee7d8-ac62-4398-ac7a-16d30f7900a2') @ [2026-10-03 21:58:12.036398]: increment.State({'count': 3, 'max_value': 3})]

In [58]:
run_object.get_final_state()

{'count': 3, 'max_value': 3}